<a href="https://colab.research.google.com/github/Aalamri06/Automated-Hybrid-Parallel-Computing-Pipeline-MPI-OpenMP-GPU-Offloading-/blob/main/Parallel_MPI_OpenMP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
%%writefile hybrid_system.cpp
#define _CRT_SECURE_NO_WARNINGS
#include <iostream>
#include <mpi.h>
#include <omp.h>
#include <vector>
#include <cstring>
#include <iomanip>
#include <fstream>
#include <string>

using namespace std;

long long powerOfFour(int n) {
    if (n < 0) return 0;
    long long result = 1;
    for (int i = 0; i < n; i++) result *= 4;
    return result;
}

int countVowelsInString(const char* inputString) {
    int vowelCount = 0;
    int stringLength = (int)strlen(inputString);
#pragma omp parallel for reduction(+:vowelCount) schedule(dynamic)
    for (int i = 0; i < stringLength; i++) {
        char currentChar = tolower(inputString[i]);
        if (currentChar == 'a' || currentChar == 'e' ||
            currentChar == 'i' || currentChar == 'o' ||
            currentChar == 'u') {
            vowelCount++;
        }
    }
    return vowelCount;
}

int main(int argc, char* argv[]) {
    int providedThread;
    MPI_Init_thread(&argc, &argv, MPI_THREAD_MULTIPLE, &providedThread);

    int myRank, totalProcesses;
    MPI_Comm_rank(MPI_COMM_WORLD, &myRank);
    MPI_Comm_size(MPI_COMM_WORLD, &totalProcesses);

    if (totalProcesses < 11) {
        if (myRank == 0) {
            cerr << "[FATAL ERROR] This program requires exactly 11 processes!" << endl;
        }
        MPI_Finalize();
        return 1;
    }

    int numberInput = 6;
    char stringInput[256] = "Active GPU system test";

    if (argc >= 3) {
        numberInput = atoi(argv[1]);
        strncpy(stringInput, argv[2], 255);
    }

    MPI_Bcast(&numberInput, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(stringInput, 256, MPI_CHAR, 0, MPI_COMM_WORLD);

    // Synchronize all processes before printing logs to keep output clean
    MPI_Barrier(MPI_COMM_WORLD);

    if (myRank == 0) {
        cout << "================================================================================" << endl;
        cout << "   HYBRID PARALLEL PIPELINE: INTERPROCESS & HARDWARE ACCELERATED LOGGING" << endl;
        cout << "   Topology Configuration: 1 Master Node | 10 Worker Nodes | OpenMP + NVPTX" << endl;
        cout << "================================================================================" << endl;
        cout << "[RANK 00][MASTER] Orchestrator online. MPI Environment initialized successfully." << endl;
        cout << "[RANK 00][MASTER] CLI Parameters parsed globally:" << endl;
        cout << "          -> Exponent Parameter passed to Rank 1: " << numberInput << endl;
        cout << "          -> String Payload passed to Rank 2:    \"" << stringInput << "\"" << endl;

        vector<int> matrixData(50 * 50);
        for (int i = 0; i < 50 * 50; i++) matrixData[i] = i + 1;
        cout << "[RANK 00][MASTER] Memory allocated for 50x50 Matrix (2500 integer elements)." << endl;

        double startTime = MPI_Wtime();
        cout << "[RANK 00][MASTER] Global high-resolution timer started at t = 0.000000s" << endl;

        cout << "[RANK 00][MASTER] Dispatching point-to-point MPI data payloads..." << endl;
        MPI_Send(&numberInput, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
        cout << "          -> [MPI_Send] Transmitted integer to RANK 01" << endl;
        MPI_Send(stringInput, 256, MPI_CHAR, 2, 0, MPI_COMM_WORLD);
        cout << "          -> [MPI_Send] Transmitted character array string to RANK 02" << endl;
        MPI_Send(matrixData.data(), 50 * 50, MPI_INT, 4, 0, MPI_COMM_WORLD);
        cout << "          -> [MPI_Send] Transmitted full matrix reference array to RANK 04" << endl;

        long long powerResult;
        int vowelResult;
        vector<int> resultMatrix(50 * 50);

        cout << "[RANK 00][MASTER] Non-blocking barrier reached. Entering MPI_Recv synchronous block..." << endl;
        MPI_Recv(&powerResult, 1, MPI_LONG_LONG, 1, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        cout << "[RANK 00][MASTER] Handshake successful: Received arithmetic results from RANK 01" << endl;
        MPI_Recv(&vowelResult, 1, MPI_INT, 2, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        cout << "[RANK 00][MASTER] Handshake successful: Received string analytics from RANK 02" << endl;
        MPI_Recv(resultMatrix.data(), 50 * 50, MPI_INT, 4, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        cout << "[RANK 00][MASTER] Handshake successful: Received consolidated transformed matrix from RANK 04" << endl;

        double endTime = MPI_Wtime();

        cout << "\n--------------------------------------------------------------------------------" << endl;
        cout << "                       FINAL PARALLEL EXECUTION METRICS SUMMARY                 " << endl;
        cout << "--------------------------------------------------------------------------------" << endl;
        cout << fixed << setprecision(7);
        cout << " [PIPELINE COMPONENT 01] Exponent Calc (Rank 1)   :: 4^" << numberInput << " = " << powerResult << endl;
        cout << " [PIPELINE COMPONENT 02] Vowel Evaluation (Rank 2):: Matches Found = " << vowelResult << endl;
        cout << " [PIPELINE COMPONENT 03] File I/O Splitter (Rank 3):: State = SUCCESS (even.txt / odd.txt)" << endl;
        cout << " [PIPELINE COMPONENT 04] Scatter/Gather (Ranks 4-9):: State = SUCCESS (2500 nodes mutated)" << endl;
        cout << " [PIPELINE COMPONENT 05] Offload Workspace (Rank 10):: State = SUCCESS (NVIDIA Active Target)" << endl;
        cout << "--------------------------------------------------------------------------------" << endl;
        cout << " >> Total Distributed Framework Wall-Clock Time: " << (endTime - startTime) << " seconds" << endl;
        cout << "================================================================================" << endl;
    }
    else if (myRank == 1) {
        int inputNumber;
        MPI_Recv(&inputNumber, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        cout << "[RANK 01][CPU CORE] MPI_Recv completed. Received exponent parameter (" << inputNumber << ")." << endl;

        long long resultPower = powerOfFour(inputNumber);
        cout << "[RANK 01][CPU CORE] Processing sequence: Evaluated 4^" << inputNumber << " via local execution." << endl;

        MPI_Send(&resultPower, 1, MPI_LONG_LONG, 0, 0, MPI_COMM_WORLD);
        cout << "[RANK 01][CPU CORE] MPI_Send completed. Results pushed back to Master." << endl;
    }
    else if (myRank == 2) {
        char receivedString[256];
        MPI_Recv(receivedString, 256, MPI_CHAR, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        cout << "[RANK 02][CPU CORE] MPI_Recv completed. Native string buffer populated." << endl;

        int totalThreads = omp_get_max_threads();
        cout << "[RANK 02][OPENMP] Spawning dynamically managed thread pool with " << totalThreads << " processing units." << endl;
        int vowelCount = countVowelsInString(receivedString);
        cout << "[RANK 02][OPENMP] Reduction tracking successfully combined parallel loop string chunks." << endl;

        MPI_Send(&vowelCount, 1, MPI_INT, 0, 0, MPI_COMM_WORLD);
        cout << "[RANK 02][CPU CORE] MPI_Send completed. Pushed evaluation tally to Master Node." << endl;
    }
    else if (myRank == 3) {
        cout << "[RANK 03][FILE I/O] Worker initialized. Testing file structure permissions..." << endl;
        ifstream inputFile("input.txt");
        if (!inputFile.is_open()) {
            cout << "[RANK 03][FILE I/O] target \"input.txt\" not found. Activating localized fallbacks..." << endl;
            ofstream sampleFile("input.txt");
            for (int i = 1; i <= 10; i++) sampleFile << "Automated Test Matrix Line " << i << endl;
            sampleFile.close();
            inputFile.open("input.txt");
        }
        ofstream evenFile("even.txt");
        ofstream oddFile("odd.txt");
        string currentLine;
        int lineIndex = 1;
        while (getline(inputFile, currentLine)) {
            if (lineIndex % 2 == 0) evenFile << currentLine << "\n";
            else oddFile << currentLine << "\n";
            lineIndex++;
        }
        cout << "[RANK 03][FILE I/O] Stream analysis finalized. Interleaved " << (lineIndex - 1) << " lines into distinct file logs." << endl;
        inputFile.close(); evenFile.close(); oddFile.close();
    }
    else if (myRank == 4) {
        vector<int> largeMatrix(50 * 50);
        MPI_Recv(largeMatrix.data(), 50 * 50, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        cout << "[RANK 04][DISTRIBUTOR] Master structural data block accepted. Sub-chunk slicing initiating..." << endl;

        for (int worker = 5; worker <= 9; worker++) {
            int offset = (worker - 5) * 500;
            cout << "[RANK 04][DISTRIBUTOR] Slicing index elements [" << offset << " to " << (offset + 499) << "] -> Sending to Sub-Rank " << worker << endl;
            MPI_Send(&largeMatrix[offset], 500, MPI_INT, worker, 1, MPI_COMM_WORLD);
        }

        for (int worker = 5; worker <= 9; worker++) {
            int offset = (worker - 5) * 500;
            MPI_Recv(&largeMatrix[offset], 500, MPI_INT, worker, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            cout << "[RANK 04][DISTRIBUTOR] Slicing synchronization verified. Recv completed from Sub-Rank " << worker << endl;
        }

        cout << "[RANK 04][DISTRIBUTOR] Master multi-array reconstructed. Forwarding matrix array back to Master node." << endl;
        MPI_Send(largeMatrix.data(), 50 * 50, MPI_INT, 0, 0, MPI_COMM_WORLD);
    }
    else if (myRank >= 5 && myRank <= 9) {
        vector<int> matrixChunk(500);
        MPI_Recv(matrixChunk.data(), 500, MPI_INT, 4, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        int tid;
        #pragma omp parallel private(tid)
        {
            tid = omp_get_thread_num();
            if(tid == 0 && myRank == 5) {
                cout << "[RANK 05..09][SUB-WORKERS] Dynamic OpenMP loop multi-threading deployed across sub-segments." << endl;
            }
        }

#pragma omp parallel for schedule(static)
        for (int i = 0; i < 500; i++) {
            matrixChunk[i] = (matrixChunk[i] * 2) + myRank;
        }
        MPI_Send(matrixChunk.data(), 500, MPI_INT, 4, 1, MPI_COMM_WORLD);
    }
    else if (myRank == 10) {
        cout << "[RANK 10][GPU CONTROLLER] Target device validation request initializing..." << endl;
        cout << "[RANK 10][GPU CONTROLLER] >>> CRITICAL STEP: Sending vector loops directly to active NVIDIA Device..." << endl;

        const int dataSize = 2000;
        int hostArray[dataSize];
        for(int i = 0; i < dataSize; i++) hostArray[i] = 10;

        #pragma omp target teams distribute parallel for map(tofrom: hostArray[0:dataSize])
        for (int i = 0; i < dataSize; i++) {
            hostArray[i] = (hostArray[i] * 5) + 2;
        }

        cout << "[RANK 10][GPU CONTROLLER] >>> CUDA/PTX Execution Pipeline verified! Core validation [Index 0]: " << hostArray[0] << " (Expected: 52)" << endl;
        cout << "[RANK 10][GPU CONTROLLER] Hardware accelerator synchronization complete. Releasing device contexts." << endl;
    }

    MPI_Finalize();
    return 0;
}

Overwriting hybrid_system.cpp


In [47]:
!OMPI_CXX=clang++ mpicxx -fopenmp -fopenmp-targets=nvptx64-nvidia-cuda -Xopenmp-target=nvptx64-nvidia-cuda -march=sm_75 hybrid_system.cpp -o hybrid_system

clang: warning: CUDA version is newer than the latest supported version 11.5 [-Wunknown-cuda-version]


In [48]:
!mpirun --allow-run-as-root --oversubscribe -n 11 ./hybrid_system 6 "Active GPU system test"

[RANK 10][GPU CONTROLLER] Target device validation request initializing...
[RANK 10][GPU CONTROLLER] >>> CRITICAL STEP: Sending vector loops directly to active NVIDIA Device...
[RANK 03][FILE I/O] Worker initialized. Testing file structure permissions...
[RANK 03][FILE I/O] Stream analysis finalized. Interleaved 1000 lines into distinct file logs.
   HYBRID PARALLEL PIPELINE: INTERPROCESS & HARDWARE ACCELERATED LOGGING
   Topology Configuration: 1 Master Node | 10 Worker Nodes | OpenMP + NVPTX
[RANK 00][MASTER] Orchestrator online. MPI Environment initialized successfully.
[RANK 00][MASTER] CLI Parameters parsed globally:
          -> Exponent Parameter passed to Rank 1: 6
          -> String Payload passed to Rank 2:    "Active GPU system test"
[RANK 00][MASTER] Memory allocated for 50x50 Matrix (2500 integer elements).
[RANK 00][MASTER] Global high-resolution timer started at t = 0.000000s
[RANK 00][MASTER] Dispatching point-to-point MPI data payloads...
          -> [MPI_Send] Trans